Transformer代码复现

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as Data
import numpy as np
import math
import time

In [ ]:
# 位置编码
class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=5000,dropout=0.1):
        super(PositionalEncoding,self).__init__()
        self.dropout=nn.Dropout(p=dropout)

        pe=torch.zeros(max_len,d_model) # 初始化位置编码
        position=torch.arange(0,max_len).unsqueeze(1).float() # 位置编码的维度
        # unsqeeze的用法，
        div_term=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
# 数值稳定性
# 填充位置编码
        pe[:,0::2]=torch.sin(position*div_term) # 偶数维度用sin
        pe[:,1::2]=torch.cos(position*div_term) # 奇数维度用cos

        pe=pe.unsqueeze(0)
        self.register_buffer('pe',pe) # 将位置编码注册为buffer，不会被更新

    def forward(self,x):

        # 前向传播
        x=x+self.pe[:,:x.size(1),:]
        return self.dropout(x)

                                                        

In [ ]:
# 多头注意力机制
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,d_k,d_v,n_heads,dropout=0.1):
        super(MultiHeadAttention,self).__init__()
        self.d_model=d_model
        self.d_k=d_k
        self.d_v=d_v
        self.n_heads=n_heads

        self.w_q=nn.Linear(d_model,d_k*n_heads)
        self.w_k=nn.Linear(d_model,d_k*n_heads)
        self.w_v=nn.Linear(d_model,d_v*n_heads)

        self.fc=nn.Linear(n_heads*d_v,d_model)
        self.dropout=nn.Dropout(dropout)
        self.layer_norm=nn.LayerNorm(d_model)
    def forward(self,Q,K,V,mask=None):
        # 前向传播  Q,K,V:[batch,seq_len,d_model]
        batch_size=Q.size(0)
        # 线性变换并分成多头
        q_s=self.w_q(Q).view(batch_size,-1, self.n_heads, self.d_k).transpose(1,2)
        k_s=self.w_k(K).view(batch_size,-1, self.n_heads,self.d_k).transpose(1,2)
        v_s=self.w_v(V).view(batch_size,-1, self.n_heads,self.d_v).transpose(1,2)
    # 计算缩放点积注意力
        scores=torch.matmul(q_s,k_s.transpose(-1,-2))/math.sqrt(self.d_k)
        if mask is not None:
            scores=scores.masked_fill(mask==0,-1e9)
        attn=torch.softmax(scores,dim=-1)
        attn=self.dropout(attn)
        # 加权求和并合并多头
        context=torch.matmul(attn,v_s)
        context=context.transpose(1,2).contiguous().view(batch_size,-1,self.n_heads*self.d_v)
        output=self.W_O(context)
        return output
 # 前馈神经网络
class PowiseFeedForward(nn.Module):
    def__ init__(self,d_model,d_ff,dropout=0.1):

